# Punto 2.2 — Experimentos: la ruta más barata por las 47 capitales

**Curso:** Redes neuronales y algoritmos bioinspirados · Universidad Nacional de Colombia

Este cuaderno **solo ejecuta los experimentos** de la colonia de hormigas (ACO) y del algoritmo genético (GA) sobre la matriz de costos en euros (tiempo del vendedor + combustible + peajes de 2026):

1. **Hiperparámetros**: todo lo que se puede cambiar está en una sola celda.
2. **Experimento 1 (híbrido)**: ACO y GA con búsqueda local, comparados con el óptimo exacto.
3. **Experimento 2 (puro)**: ACO y GA clásicos, sin búsqueda local.

Cada experimento muestra la ruta encontrada y el GIF de la evolución de la mejor solución sobre el mapa.

**Cómo usarlo.** Cambia los valores de la celda de hiperparámetros y ejecuta todas las celdas (*Run All*, ~1 min). Todos los datos están en `data/`: no se necesita internet.

**Dónde está el resto.** La explicación de los datos, el modelo de costo y el estudio de sensibilidad al valor de la hora están en el informe (`docs/index.html`). El barrido de sensibilidad se ejecuta desde la terminal: `python3 src/sensibilidad.py`.

Ejemplo para el video

Cada ruta cuesta: valor de la hora × horas + combustible + peaje. El combustible es 0,0846 € por km.

Datos de A Coruña → Pontevedra:
- Con peaje (AP-9): 133,5 km, 1,452 h (1 h 27 min). Combustible: 133,5 × 0,0846 = 11,29 €. Peaje: 14,18 €.
- Sin peaje: 132,4 km, 2,420 h (2 h 25 min). Combustible: 132,4 × 0,0846 = 11,20 €. Peaje: 0 €.

Si la hora vale 10 €:
- con peaje: 10 × 1,452 = 14,52 → 14,52 + 11,29 + 14,18 = 40,00 €
- sin peaje: 10 × 2,420 = 24,20 → 24,20 + 11,20 + 0 = 35,40 € → conviene la carretera gratuita

Si la hora vale 26,18 € (INE):
- con peaje: 26,18 × 1,452 = 38,02 → 38,02 + 11,29 + 14,18 = 63,50 €
- sin peaje: 26,18 × 2,420 = 63,35 → 63,35 + 11,20 + 0 = 74,55 € → conviene pagar la AP-9

In [ ]:
# Preparación: rutas, librerías y módulos del proyecto
from pathlib import Path
import copy
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

PARTE_2 = Path.cwd()
if not (PARTE_2 / "src" / "tsp_espana.py").exists():          # por si se abre desde la raíz
    PARTE_2 = next(p for p in [Path.cwd() / "punto_2" / "parte_2", Path.cwd().parent / "parte_2"]
                   if (p / "src" / "tsp_espana.py").exists())
sys.path.insert(0, str(PARTE_2 / "src"))

import tsp_espana as te          # datos, modelo de costo, ACO/GA de la parte 1, óptimo exacto, mapas

pd.set_option("display.max_colwidth", 120)
datos = te.cargar_datos()
NOMBRES = datos["names"]
print(f"Carpeta del proyecto: {PARTE_2}")
print(f"Capitales: {len(NOMBRES)} · pares ordenados: {len(NOMBRES) * (len(NOMBRES) - 1)}")

## 1. Hiperparámetros

Después de modificar esta celda, vuelve a ejecutar las celdas siguientes (o *Run All*).

| Parámetro | Qué controla |
|---|---|
| `VALOR_HORA` | €/h del vendedor. Valor base: coste laboral por hora del INE, 2.º trimestre de 2026 |
| `PRECIO_COMBUSTIBLE`, `CONSUMO` | €/L de gasolina 95 y L/100 km del Toyota Corolla Hybrid 140H |
| `ACO_HIBRIDO`, `GA_HIBRIDO` | Experimento 1, con búsqueda local 2-opt + Or-opt |
| `ACO_PURO`, `GA_PURO` | Experimento 2, ACO y GA clásicos |
| `GENERAR_GIF` | Animación de la evolución de la mejor solución (tarda unos segundos más) |

Parámetros del ACO: `alpha` (peso de la feromona), `beta` (peso del costo del tramo, $1/C_{ij}$), `rho` (evaporación), `elitismo` (refuerzo de la mejor ruta global). Parámetros del GA: `p_cruce` (cruce OX), `p_mutacion` (mutación por inversión), `p_busqueda_local` (fracción de hijos pulidos), `n_elite`, `k_torneo`.

> En el experimento 1 la búsqueda local suele llevar a los dos algoritmos al óptimo, así que el efecto de un hiperparámetro se ve mejor en el **experimento 2** (por ejemplo, `generaciones=300` en `GA_PURO` o `beta=1.0` en `ACO_PURO`).
>
> Al ejecutar, los resultados se guardan en `results/hibrido/` y `results/puro/`, que también usa el informe. Si cambias hiperparámetros, vuelve a los valores base y ejecuta otra vez (o `git checkout -- punto_2/parte_2/results`) para restaurarlos.

In [ ]:
# ─────────────────────────── HIPERPARÁMETROS ───────────────────────────
SEMILLA = 20261008               # misma semilla => mismo resultado
INICIO = "Madrid"                # solo cambia desde dónde se MUESTRA el ciclo

# Costos
VALOR_HORA = 26.18               # €/h  (INE, coste laboral por hora efectiva, 2T 2026)
PRECIO_COMBUSTIBLE = 1.799       # €/L  (gasolina 95, 5-oct-2026)
CONSUMO = 4.7                    # L/100 km (Toyota Corolla Hybrid 140H, WLTP)

# Experimento 1: híbrido (con búsqueda local)
ACO_HIBRIDO = dict(n_hormigas=20, iteraciones=200, alpha=1.0, beta=3.0, rho=0.1,
                   elitismo=5.0, busqueda_local=True)
GA_HIBRIDO = dict(tam_poblacion=150, generaciones=300, p_cruce=0.9, p_mutacion=0.3,
                  p_busqueda_local=0.05, n_elite=2, k_torneo=3)

# Experimento 2: puro (sin búsqueda local)
ACO_PURO = dict(ACO_HIBRIDO, n_hormigas=47, iteraciones=300, busqueda_local=False)
GA_PURO = dict(GA_HIBRIDO, generaciones=1500, p_busqueda_local=0.0)

GENERAR_GIF = True               # animación de la evolución de la mejor ruta
# ────────────────────────────────────────────────────────────────────────


def configurar(modo, aco, ga):
    # Configuración completa con los valores de esta celda (mismo formato que config.ini)
    cfg = copy.deepcopy(te.POR_DEFECTO)
    cfg["general"].update(semilla=SEMILLA, inicio=INICIO, generar_gif=GENERAR_GIF, modo=modo,
                          algoritmo="both", verificar_exacto=True)
    cfg["costos"].update(valor_hora=VALOR_HORA, precio_combustible=PRECIO_COMBUSTIBLE,
                         consumo_l_100km=CONSUMO)
    cfg["aco"].update(aco)
    cfg["ga"].update(ga)
    te.validar_config(cfg, NOMBRES)
    return cfg


print(f"Combustible: {CONSUMO / 100 * PRECIO_COMBUSTIBLE:.4f} €/km · valor de la hora: {VALOR_HORA} €/h")

## 2. Experimento 1 — ACO y GA con búsqueda local (híbridos)

Los algoritmos son los de la parte 1 (`punto_2/parte_1/tsp_bio.py`), aplicados a la matriz de costos en euros. El **óptimo exacto** (programación lineal entera) sirve de referencia: `gap_optimo_pct` es la distancia de cada algoritmo a él.

In [ ]:
t0 = time.time()
hib = te.ejecutar(configurar("hibrido", ACO_HIBRIDO, GA_HIBRIDO), datos, verbose=False)
print(f"Tiempo total: {time.time() - t0:.0f} s · óptimo exacto: {hib['optimo']:,.2f} €")


def tabla_resumen(r, etiqueta):
    cols = ["algoritmo", "costo_total_eur", "gap_optimo_pct", "km", "horas", "tiempo_eur",
            "combustible_eur", "peaje_eur", "tramos_con_peaje", "iteracion_mejor", "segundos"]
    t = r["resumen"][cols].copy()
    t.insert(0, "experimento", etiqueta)
    return t

display(tabla_resumen(hib, "híbrido"))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for ax, alg in zip(axs, ["aco", "ga"]):
    te.grafico_convergencia(hib["resultados"][alg]["historial"], alg, VALOR_HORA, hib["optimo"], ax=ax)
fig.tight_layout()
plt.show()

### Ruta encontrada

Tramos en los que conviene pagar peaje (en rojo en el mapa):

In [ ]:
ruta = hib["resultados"]["aco"]["tabla"]
display(ruta[ruta["variante"] == "con peaje"][["desde", "hasta", "vias_peaje", "km", "horas",
                                               "peaje_eur", "costo_eur"]].round(2))
display(Image(filename=str(hib["carpeta"] / "mapa_ruta_aco.png"), width=720))

In [ ]:
# Orden completo, tramo a tramo
hib["resultados"]["aco"]["tabla"][["orden", "desde", "hasta", "variante", "km", "horas", "tiempo_eur",
                                   "combustible_eur", "peaje_eur", "costo_eur",
                                   "costo_acumulado_eur"]].round(2)

### Evolución de la mejor solución (híbrido)

Un cuadro por cada mejora de la mejor ruta (rojo: tramo con peaje; azul: red libre). Con búsqueda local se llega al óptimo en pocas iteraciones, así que hay pocos cuadros.

In [ ]:
if GENERAR_GIF:
    for alg in ["ga", "aco"]:
        display(Markdown(f"**{alg.upper()} híbrido**"))
        display(Image(filename=str(hib["carpeta"] / f"evolucion_{alg}.gif"), width=640))

## 3. Experimento 2 — ACO y GA clásicos (puros)

Sin búsqueda local se ve la capacidad propia de cada metaheurística y el efecto de sus hiperparámetros sobre el recorrido.

In [ ]:
t0 = time.time()
pur = te.ejecutar(configurar("puro", ACO_PURO, GA_PURO), datos, verbose=False)
print(f"Tiempo total: {time.time() - t0:.0f} s")
display(pd.concat([tabla_resumen(hib, "híbrido"), tabla_resumen(pur, "puro")], ignore_index=True))

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for ax, alg in zip(axs, ["aco", "ga"]):
    te.grafico_convergencia(pur["resultados"][alg]["historial"], alg, VALOR_HORA, pur["optimo"], ax=ax)
fig.tight_layout()
plt.show()

### Ruta encontrada por cada algoritmo

In [ ]:
for alg in ["aco", "ga"]:
    t = pur["resultados"][alg]["tabla"]
    display(Markdown(f"**{alg.upper()} puro**: {' → '.join(list(t['desde']) + [t['hasta'].iloc[-1]])}"))
    display(Image(filename=str(pur["carpeta"] / f"mapa_ruta_{alg}.png"), width=720))

### Evolución de la mejor solución (puro)

Un cuadro por cada mejora de la mejor ruta (rojo: tramo con peaje; azul: red libre).

In [ ]:
if GENERAR_GIF:
    for alg in ["ga", "aco"]:
        display(Markdown(f"**{alg.upper()} puro**"))
        display(Image(filename=str(pur["carpeta"] / f"evolucion_{alg}.gif"), width=640))